<a href="https://colab.research.google.com/github/russianoracle/punct-nlu-colab-training/blob/main/PunctNLU_Colab_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PunctNLU CUDA Training on Colab
Training Russian punctuation restoration model on GPU

**Model:** rubert-tiny2 (29.2M params) → 12-class punctuation + capitalization

**Corpus:** 1.194M Russian sentences from Tatoeba

**Hardware:** Colab GPU (T4/A100)

## Step 1: Setup & Install Dependencies

In [ ]:
!pip install -q torch transformers numpy pandas scikit-learn tqdm

In [ ]:
import logging
import re
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Iterator

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)

# Check GPU
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Hugging Face Authentication (Optional) ────────────────────────────────────
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        import os
        os.environ['HF_TOKEN'] = hf_token
        print("✓ HF_TOKEN loaded from Colab Secrets")
except:
    print("⚠️  HF_TOKEN not found in Colab Secrets (optional, will use lower rate limits)")
    print("   To add: Click 🔑 Secrets → Add HF_TOKEN")

In [ ]:
from google.colab import drive

# Optional: Mount Drive only if you need to save results there
# drive.mount('/content/drive', force_remount=True)
print("✓ Colab environment ready")

In [ ]:
from google.colab import drive, files
from pathlib import Path
import shutil
import os

print("=" * 80)
print("CORPUS DETECTION")
print("=" * 80)

# Check if corpus already exists in /content/
CORPUS_PATH = Path("/content/sentences.csv")
print(f"1️⃣  Checking /content/sentences.csv... {CORPUS_PATH.exists()}")

if CORPUS_PATH.exists():
    size_mb = CORPUS_PATH.stat().st_size / 1e6
    print(f"   ✓ Found: {size_mb:.0f} MB")
else:
    # Mount Google Drive as fallback
    print("2️⃣  /content not found, mounting Google Drive...")
    drive.mount('/content/drive', force_remount=True)
    
    # Create directory for corpus
    corpus_dir = Path("/content/drive/MyDrive/PunctNLU")
    corpus_dir.mkdir(parents=True, exist_ok=True)
    print(f"   Created: {corpus_dir}")
    
    CORPUS_PATH = corpus_dir / "sentences.csv"
    print(f"3️⃣  Checking Google Drive... {CORPUS_PATH.exists()}")
    
    # Check if corpus exists in Drive
    if CORPUS_PATH.exists():
        size_mb = CORPUS_PATH.stat().st_size / 1e6
        print(f"   ✓ Found: {size_mb:.0f} MB")
    else:
        print("4️⃣  Not in Drive either, prompting for upload...")
        print("   Click 'Choose Files' below and select sentences.csv")
        
        uploaded = files.upload()
        if 'sentences.csv' in uploaded:
            shutil.move('./sentences.csv', str(CORPUS_PATH))
            size_mb = CORPUS_PATH.stat().st_size / 1e6
            print(f"   ✓ Uploaded: {size_mb:.0f} MB")

print(f"\nFinal CORPUS_PATH: {CORPUS_PATH}")
print(f"Type: {type(CORPUS_PATH)}")
print(f"Exists: {CORPUS_PATH.exists()}")
print("=" * 80)

### 👆 Click above cell to mount Drive and upload corpus
1. Run cell above
2. If corpus not found, click "Choose Files" button
3. Select `sentences.csv` from your computer
4. Colab will upload it to Google Drive automatically (first run only ~5-10 min)
5. Run remaining cells for training!
✅ Notebook handles everything else

## Step 3: Configuration (CUDA-Optimized)

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────

MODEL_ID      = "cointegrated/rubert-tiny2"
SEQ_LEN       = 64
NUM_LABELS    = 12
BATCH_SIZE    = 128  # Higher on GPU (was 16 on M2)
NUM_EPOCHS    = 3
LEARNING_RATE = 1e-4
WARMUP_STEPS  = 500
WEIGHT_DECAY  = 1e-5
NUM_WORKERS   = 2
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR    = Path("./checkpoints/punct_nlu")

PUNCT_CHARS   = {",": 1, ".": 2, "!": 2, "…": 2, ";": 2, "?": 3}

# CORPUS_PATH is already set in cell 6 (mounted from Google Drive)
# Ensure it's a Path object
if isinstance(CORPUS_PATH, str):
    CORPUS_PATH = Path(CORPUS_PATH)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("\n" + "=" * 80)
print("CONFIGURATION")
print("=" * 80)
log.info(f"Device: {DEVICE}")
log.info(f"Batch Size: {BATCH_SIZE}")
log.info(f"Model: {MODEL_ID}")
log.info(f"Corpus: {CORPUS_PATH}")
log.info(f"Corpus exists: {CORPUS_PATH.exists()}")
log.info(f"Output dir: {OUTPUT_DIR}")
print("=" * 80)

## Step 4: Data Loading & Processing

In [ ]:
# ── Label extraction ───────────────────────────────────────────────────────
from typing import Optional

def cap_mode_vec(words: np.ndarray) -> np.ndarray:
    """Vectorized capitalization detection (faster than list comp)"""
    modes = np.zeros(len(words), dtype=np.int32)
    for i, word in enumerate(words):
        clean = re.sub(r"[^\w]", "", word, flags=re.UNICODE)
        if clean:
            alpha = [c for c in clean if c.isalpha()]
            if alpha:
                if all(c.isupper() for c in alpha):
                    modes[i] = 2
                elif alpha[0].isupper():
                    modes[i] = 1
    return modes


def punct_class_vec(words: np.ndarray) -> np.ndarray:
    """Vectorized punctuation detection (faster than list comp)"""
    classes = np.zeros(len(words), dtype=np.int32)
    for i, word in enumerate(words):
        if word:
            last = word[-1]
            classes[i] = PUNCT_CHARS.get(last, 0)
    return classes


def word_labels_vec(words: list[str]) -> np.ndarray:
    """Vectorized label creation from word list"""
    words_arr = np.array(words, dtype=object)
    punct_classes = punct_class_vec(words_arr)
    cap_modes = cap_mode_vec(words_arr)
    return punct_classes * 3 + cap_modes


def ortho_to_norm(word: str) -> str:
    return re.sub(r"[^\w]", "", word, flags=re.UNICODE).lower()

In [ ]:
# ── Data loader ────────────────────────────────────────────────────────────

def iter_tatoeba(path: Path) -> Iterator[tuple[list[str], list[int]]]:
    """Read Tatoeba corpus without limits"""
    count = 0
    with path.open(encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t", 2)
            if len(parts) != 3 or parts[1] != "rus":
                continue

            sentence = parts[2].strip()
            if len(sentence) < 5:
                continue

            words = sentence.split()
            if len(words) < 2:
                continue

            labels = word_labels_vec(words).tolist()
            norms = [ortho_to_norm(w) for w in words]

            if not any(norms):
                continue

            yield norms, labels
            count += 1
            if count % 100000 == 0:
                log.info(f"  Loaded {count} sentences...")


@dataclass
class EncodedSample:
    input_ids: torch.Tensor      # [SEQ_LEN]
    attention_mask: torch.Tensor # [SEQ_LEN]
    labels: torch.Tensor         # [SEQ_LEN]


def encode(norm_words: list[str], word_labels: list[int], tokenizer) -> Optional[EncodedSample]:
    """Encode sentence with proper word-start alignment"""
    ids = np.zeros(SEQ_LEN, dtype=np.int32)
    mask = np.zeros(SEQ_LEN, dtype=np.int32)
    labels = np.full(SEQ_LEN, -100, dtype=np.int32)

    cls_id = tokenizer.cls_token_id
    sep_id = tokenizer.sep_token_id

    ids[0] = cls_id
    mask[0] = 1
    pos = 1

    for word, lbl in zip(norm_words, word_labels):
        if pos >= SEQ_LEN - 1:
            break

        sub_ids = tokenizer.encode(word, add_special_tokens=False)
        if not sub_ids:
            sub_ids = [tokenizer.unk_token_id]

        if pos + len(sub_ids) >= SEQ_LEN:
            break

        ids[pos] = sub_ids[0]
        mask[pos] = 1
        labels[pos] = lbl
        pos += 1

        for sub_id in sub_ids[1:]:
            if pos >= SEQ_LEN:
                break
            ids[pos] = sub_id
            mask[pos] = 1
            labels[pos] = -100
            pos += 1

    if pos < SEQ_LEN:
        ids[pos] = sep_id
        mask[pos] = 1
        pos += 1

    return EncodedSample(
        input_ids=torch.from_numpy(ids).long(),
        attention_mask=torch.from_numpy(mask).long(),
        labels=torch.from_numpy(labels).long(),
    )


class PunctDataset(Dataset):
    def __init__(self, samples: list[EncodedSample]):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        return {
            "input_ids": s.input_ids,
            "attention_mask": s.attention_mask,
            "labels": s.labels,
        }

## Step 5: Model Definition

In [ ]:
# ── Model ──────────────────────────────────────────────────────────────────

class PunctNLUModel(nn.Module):
    """Two-head model: punct restoration + actionable classification"""

    def __init__(self, model_id: str, num_labels: int = 12):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_id)
        self.dropout = nn.Dropout(0.1)

        # Head A1: Punctuation restoration (12 classes)
        self.punct_head = nn.Linear(self.bert.config.hidden_size, num_labels)

        # Head A2: Actionable classification (2 classes)
        self.classify_head = nn.Linear(self.bert.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state  # [B, L, H]
        hidden = self.dropout(hidden)

        punct_logits = self.punct_head(hidden)      # [B, L, 12]
        classify_logits = self.classify_head(hidden) # [B, L, 2]

        return punct_logits, classify_logits

## Step 6: Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, device, scaler=None):
    """Train for one epoch with Mixed Precision Training"""
    model.train()
    total_loss = 0.0
    batch_losses = []

    ce_loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

    for batch_idx, batch in enumerate(loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        # Mixed Precision: forward pass in FP16
        if scaler is not None:
            with torch.cuda.amp.autocast():
                punct_logits, classify_logits = model(input_ids, attention_mask)

                loss_punct = ce_loss_fn(punct_logits.view(-1, 12), labels.view(-1))
                
                # Binary classification: has punctuation (label > 0) or not (label == 0)
                binary_labels = (labels > 0).long()
                loss_classify = ce_loss_fn(
                    classify_logits.view(-1, 2),
                    binary_labels.view(-1)
                )

                loss = loss_punct + 0.5 * loss_classify

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            # Fallback to FP32 if no scaler
            punct_logits, classify_logits = model(input_ids, attention_mask)

            loss_punct = ce_loss_fn(punct_logits.view(-1, 12), labels.view(-1))
            
            # Binary classification: has punctuation (label > 0) or not (label == 0)
            binary_labels = (labels > 0).long()
            loss_classify = ce_loss_fn(
                classify_logits.view(-1, 2),
                binary_labels.view(-1)
            )

            loss = loss_punct + 0.5 * loss_classify
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        scheduler.step()

        loss_val = loss.item()
        total_loss += loss_val
        batch_losses.append(loss_val)
        
        # GPU memory tracking
        if torch.cuda.is_available():
            gpu_mem = torch.cuda.memory_allocated() / 1e9
        else:
            gpu_mem = 0

        if (batch_idx + 1) % 100 == 0:
            avg_loss = np.mean(batch_losses[-100:])
            lr = optimizer.param_groups[0]['lr']
            log.info(f"  Batch {batch_idx + 1:4d}/{len(loader)}  "
                    f"loss={loss_val:.4f}  avg_loss={avg_loss:.4f}  "
                    f"gpu_mem={gpu_mem:.2f}GB  lr={lr:.2e}")

    return total_loss / len(loader)

def eval_epoch(model, loader, device):
    """Evaluate for one epoch with detailed logging"""
    model.eval()
    total_loss = 0.0
    correct = torch.tensor(0, device=device)
    total = torch.tensor(0, device=device)
    ce_loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

    with torch.no_grad():
        for batch_idx, batch in enumerate(loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            punct_logits, classify_logits = model(input_ids, attention_mask)

            loss_punct = ce_loss_fn(punct_logits.view(-1, 12), labels.view(-1))
            
            # Binary classification: has punctuation (label > 0) or not (label == 0)
            binary_labels = (labels > 0).long()
            loss_classify = ce_loss_fn(
                classify_logits.view(-1, 2),
                binary_labels.view(-1)
            )

            loss = loss_punct + 0.5 * loss_classify
            total_loss += loss.item()

            pred = torch.argmax(punct_logits, dim=-1)
            mask = labels >= 0
            correct += (pred[mask] == labels[mask]).sum()
            total += mask.sum()
    accuracy = (correct / max(total, 1)).item()
    avg_loss = total_loss / len(loader)
    
    return avg_loss, accuracy

## Step 7: Main Training Function

In [ ]:
# ── GPU Memory Tracker ────────────────────────────────────────────────────────
def get_gpu_memory():
    """Get GPU memory usage in GB"""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1e9, torch.cuda.get_device_properties(0).total_memory / 1e9
    return 0, 0

def log_gpu_usage(label=""):
    """Log current GPU memory usage"""
    used, total = get_gpu_memory()
    if total > 0:
        pct = 100 * used / total
        log.info(f"{label:20s} GPU: {used:6.2f} / {total:6.2f} GB ({pct:5.1f}%)")

def main():
    log.info("=" * 100)
    log.info("🚀 STARTING TRAINING")
    log.info("=" * 100)
    
    # Force device to CUDA if available
    global DEVICE
    if torch.cuda.is_available():
        DEVICE = "cuda"
        torch.cuda.set_device(0)
        torch.cuda.empty_cache()
        log.info("✓ CUDA forced ON, cache cleared")
    else:
        log.error("✗ CUDA not available! Training will be on CPU (very slow)")
    
    log_gpu_usage("Initial state:")
    log.info("=" * 100)
    log.info("🚀 STARTING TRAINING")
    log.info("=" * 100)

    # ── 1. Load corpus ──────────────────────────────────────────────────────
    log.info(f"\n📂 Loading corpus from {CORPUS_PATH}...")
    if not CORPUS_PATH.exists():
        log.error(f"✗ Corpus not found: {CORPUS_PATH}")
        return False

    corpus_data = list(iter_tatoeba(CORPUS_PATH))
    log.info(f"✓ Loaded {len(corpus_data):,} sentences")

    # ── 2. Load tokenizer ───────────────────────────────────────────────────
    log.info(f"\n🔤 Loading tokenizer: {MODEL_ID}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, timeout=60)
    log.info(f"✓ Tokenizer ready: vocab_size={len(tokenizer)}")

    # ── 3. Encode all samples ───────────────────────────────────────────────
    log.info(f"\n⚙️  Encoding {len(corpus_data):,} samples...")
    t0 = time.time()
    encoded_samples = []
    for idx, (norm_words, labels) in enumerate(corpus_data):
        sample = encode(norm_words, labels, tokenizer)
        if sample:
            encoded_samples.append(sample)
        if (idx + 1) % 100000 == 0:
            elapsed = time.time() - t0
            rate = (idx + 1) / elapsed
            log.info(f"  {idx + 1:,} / {len(corpus_data):,}  "
                    f"({100*(idx+1)//len(corpus_data)}%)  "
                    f"rate={rate:.0f} samples/sec")

    elapsed = time.time() - t0
    log.info(f"✓ Encoded {len(encoded_samples):,} samples in {elapsed:.1f}s")

    # ── 4. Split train/val ──────────────────────────────────────────────────
    log.info(f"\n📊 Splitting train/val...")
    np.random.seed(42)
    indices = np.random.permutation(len(encoded_samples))
    split = int(0.95 * len(encoded_samples))

    train_indices = indices[:split]
    val_indices = indices[split:]

    train_dataset = PunctDataset([encoded_samples[i] for i in train_indices])
    val_dataset = PunctDataset([encoded_samples[i] for i in val_indices])

    log.info(f"✓ Train: {len(train_dataset):,}  Val: {len(val_dataset):,}  "
            f"(ratio={len(train_dataset)//len(val_dataset)}:1)")

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
        persistent_workers=False,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        num_workers=0,
        pin_memory=True,
        persistent_workers=False,
    )
    
    log.info(f"✓ DataLoaders: {len(train_loader)} train batches, {len(val_loader)} val batches")

    # ── 5. Initialize model ─────────────────────────────────────────────────
    log_gpu_usage("Before model:")
    log.info(f"\n🧠 Building model: {MODEL_ID}")
    model = PunctNLUModel(MODEL_ID).to(DEVICE)
    params = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log_gpu_usage("After model load:")
    # Verify model is on GPU
    assert next(model.parameters()).is_cuda, "ERROR: Model is on CPU!"
    log.info(f"✓ Model on {DEVICE}  Params: {params:,}  Trainable: {trainable:,}")

    # ── 6. Training loop ────────────────────────────────────────────────────
    log.info(f"\n⚡ Training config:")
    log.info(f"  Device: {DEVICE}")
    log.info(f"  Batch size: {BATCH_SIZE}")
    log.info(f"  Learning rate: {LEARNING_RATE}")
    log.info(f"  Epochs: {NUM_EPOCHS}")
    
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    total_steps = len(train_loader) * NUM_EPOCHS
    scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=1.0,
        end_factor=0.0,
        total_iters=total_steps,
    )

    best_val_loss = float("inf")
    training_log = []

    # Initialize GradScaler for Mixed Precision Training
    scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None
    log.info(f"✓ Mixed Precision Training: {'ENABLED (AMP)' if scaler else 'DISABLED (CPU)'}")

    log.info(f"\n{'='*100}")
    log.info(f"EPOCH TRAINING")
    log.info(f"{'='*100}")

    for epoch in range(1, NUM_EPOCHS + 1):
        t0 = time.time()

        train_loss = train_epoch(model, train_loader, optimizer, scheduler, DEVICE, scaler)
        val_loss, val_acc = eval_epoch(model, val_loader, DEVICE)

        elapsed = time.time() - t0
        
        log.info(f"\n📈 Epoch {epoch}/{NUM_EPOCHS}")
        log.info(f"  Train loss: {train_loss:.4f}  Val loss: {val_loss:.4f}  "
                f"Val acc: {val_acc:.3f}  Time: {elapsed:.0f}s")

        training_log.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_acc': val_acc,
            'time': elapsed
        })

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), OUTPUT_DIR / "best.pt")
            log.info(f"  ✓ New best model saved (val_loss={val_loss:.4f})")

    # Final summary
    log.info(f"\n{'='*100}")
    log.info(f"TRAINING COMPLETE")
    log.info(f"{'='*100}")
    log.info(f"Best validation loss: {best_val_loss:.4f}")
    log.info(f"Total training time: {sum(entry['time'] for entry in training_log):.0f}s")
    log.info(f"Average epoch time: {np.mean([entry['time'] for entry in training_log]):.0f}s")

    # Save final checkpoint
    torch.save({
        "model_state_dict": model.state_dict(),
        "best_val_loss": best_val_loss,
        "training_log": training_log,
    }, OUTPUT_DIR / "punct_nlu.pt")
    log.info(f"✓ Final checkpoint saved → {OUTPUT_DIR / 'punct_nlu.pt'}")
    log.info(f"{'='*100}\n")

    return True

## Step 8: RUN TRAINING

In [ ]:
# ── GPU Recovery & Diagnostic ──────────────────────────────────────────────────

print("\n" + "=" * 80)
print("GPU RECOVERY & DIAGNOSTIC")
print("=" * 80)

print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
print(f"torch.cuda.device_count(): {torch.cuda.device_count()}")
print(f"torch.cuda.get_device_name(0): {torch.cuda.get_device_name(0)}")
print(f"torch.version.cuda: {torch.version.cuda}")

# Clear GPU cache
print("\n🔄 Clearing GPU cache...")
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Try to allocate a small tensor on GPU
print("Testing GPU allocation...")
try:
    test_tensor = torch.zeros(1, device="cuda")
    print(f"✓ GPU allocation successful")
    print(f"  Test tensor device: {test_tensor.device}")
    del test_tensor
    torch.cuda.empty_cache()
except Exception as e:
    print(f"✗ GPU allocation failed: {e}")
    print("\n💡 SOLUTIONS:")
    print("  1. Runtime → Restart runtime (clears GPU)")
    print("  2. Or reduce BATCH_SIZE in config (Cell 9)")
    print("  3. Or use CPU by setting DEVICE='cpu'")
    raise

# Check model will be on GPU
print(f"\nConfigured device: {DEVICE}")
print(f"GPU memory available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("=" * 80 + "\n")

print("\n" + "=" * 80)
print("STARTING TRAINING")
print("=" * 80)
print(f"Corpus path: {CORPUS_PATH}")
print(f"Corpus exists: {CORPUS_PATH.exists()}")
print(f"Device: {DEVICE}")
print("=" * 80 + "\n")

success = main()
if success:
    print("\n" + "=" * 80)
    print("✓ Training completed successfully!")
    print("=" * 80)
else:
    print("\n" + "=" * 80)
    print("✗ Training failed. Check logs above.")
    print("=" * 80)

## Step 9: Download Results

In [ ]:
from google.colab import files
from pathlib import Path

# Check if training completed
checkpoint_dir = Path("./checkpoints/punct_nlu")
best_model = checkpoint_dir / "best.pt"
final_model = checkpoint_dir / "punct_nlu.pt"

if best_model.exists():
    print(f"Downloading trained model: {best_model.stat().st_size / 1e6:.0f} MB")
    files.download(str(best_model))
    print("✓ Downloaded: best.pt")
elif final_model.exists():
    print(f"Downloading final checkpoint: {final_model.stat().st_size / 1e6:.0f} MB")
    files.download(str(final_model))
    print("✓ Downloaded: punct_nlu.pt")
else:
    print("✗ Training output not found!")
    print(f"Checked: {checkpoint_dir}")
    print("
Did training complete successfully?")
    print("Check the training log above for errors.")